[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/two-phaseflow/JED116F/blob/main/notebooks/04_lumped_models_inverse.ipynb)

Click the badge to open this notebook in Google Colab. In Colab, choose **File > Save a copy in Drive** before you start, so your work is kept.

# JEÐ116F Notebook 04. Lumped models and inverse modelling

Lecture 9, Monday 2 November 2026. Parts 1 to 3 are started in class, in blocks between the board sessions. The completed notebook is part of Assignment 4; the last section says what to hand in.

**Cell types**

- **A. Run and look.** The code is complete. Run it and read the output.
- **B. Fill the gaps.** Replace every `___` with a short expression. The `assert` test after each gap checks your line.
- **C. Extensions.** Open questions for when you finish early.

Before each computation, write your prediction in the cell marked *Prediction*.

**Rules for this course**

1. Keep **units** next to every number in your code comments.
2. Nothing physical is typed in by hand when a property package can give it: water properties come from IAPWS-IF97 (`iapws`).
3. You may use AI tools for error messages, plotting and coding ideas. You must be able to explain every line you hand in.

**Parts**

1. From rings to tanks: the mass balance of a few tanks, the same balance as the ring model of notebook 02
2. Inverse modelling: fit a tank model to the 1997 production test at Hofsstaðir
3. The Hofsstaðir record 1997 to 2007: which model, which forecast

**Assignment 4** at the end: what to hand in.

**Symbols used throughout**

| symbol | meaning | unit |
|---|---|---|
| $W$ | production, mass flow out of the well | kg/s |
| $P,\ P_i$ | pressure; pressure in tank $i$ (tank 1 holds the well) | Pa |
| $P_0$ | initial pressure, and the boundary pressure of an open model | Pa |
| $S_m$ | mass storativity | kg/(m² Pa) |
| $A_i,\ S_m A_i$ | area of tank $i$, storage of tank $i$ | m², kg/Pa |
| $\sigma_i$ | conductance between tank $i$ and tank $i+1$ | kg/(s Pa), or (kg/s)/bar |
| $w_i$ | mass flow from tank $i+1$ into tank $i$ | kg/s |
| $T = kh/\nu$ | transmissivity | m s |
| $\tau = S_m A/\sigma$ | time scale of an open tank | s |
| $\rho,\ g$ | water density, gravitational acceleration | kg/m³, m/s² |
| $\phi,\ c_t,\ h$ | porosity, total compressibility, reservoir thickness | -, 1/Pa, m |
| RMS, $s$ | misfit, standard deviation of the data about the model | Pa, or m of water |
| $n,\ p$ | number of data points, number of model parameters | |

## Part 0. Setup

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.special import exp1

try:
    from iapws import IAPWS97
except ImportError:
    !pip -q install iapws
    from iapws import IAPWS97

# data files: read from ./data if present, otherwise download from the course repository
BASE = "https://raw.githubusercontent.com/two-phaseflow/JED116F/main/data/"
FILES = ["Hofsstadir_lumped_1997-2007.csv", "Ahuachapan_lumped_1968-2001.csv", "Gata_lumped_1982-2000.csv"]
os.makedirs("data", exist_ok=True)
for f in FILES:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve(BASE + f, "data/" + f)

plt.rcParams.update({"font.size": 12, "axes.grid": True})

In [ ]:
# units and constants
minute, hour, day = 60.0, 3600.0, 86400.0
yr = 365.25 * day
bar = 1e5          # Pa
g = 9.81           # m/s2

def water(P_MPa, T_C):
    '''Liquid water density rho [kg/m3] and kinematic viscosity nu [m2/s] from IAPWS-IF97.'''
    w = IAPWS97(P=P_MPa, T=T_C + 273.15)
    return w.rho, w.mu / w.rho

rho90, nu90 = water(2.0, 90.0)
print(f"water at 2 MPa and 90 C: rho = {rho90:.1f} kg/m3, nu = {nu90:.3e} m2/s")

## Part 1. From rings to tanks

In notebook 02 you wrote the mass balance of one ring around the well:

$$S_m \, 2\pi r \Delta r \, \frac{dP}{dt} = w_{\mathrm{out}} - w_{\mathrm{in}}$$

- $S_m$: mass storativity [kg/(m² Pa)]
- $2\pi r \Delta r$: area of the ring [m²]
- $P$: pressure in the ring [Pa]
- $t$: time [s]
- $w_{\mathrm{out}}$: mass flow into the ring through its outer face [kg/s]
- $w_{\mathrm{in}}$: mass flow out of the ring through its inner face, toward the well [kg/s]

A lumped model uses the same balance on a few large volumes, called tanks. Tank 1 holds the well. Tank $i$ has area $A_i$, so its storage is $S_m A_i$:

$$S_m A_i \, \frac{dP_i}{dt} = w_i - w_{i-1}$$

- $S_m A_i$: storage of tank $i$ [kg/Pa], the mass that enters the tank when its pressure rises by 1 Pa
- $P_i$: pressure in tank $i$ [Pa]
- $w_i$: mass flow from tank $i+1$ into tank $i$ [kg/s]
- $w_0 = W$: the production from the well in tank 1 [kg/s]

Darcy between two tanks:

$$w_i = \sigma_i \,(P_{i+1} - P_i)$$

- $\sigma_i$: conductance between tank $i$ and tank $i+1$ [kg/(s Pa)]; can be given in (kg/s)/bar.

**Open model:** the last tank is joined through one more conductance to a boundary that stays at the initial pressure $P_0$. This is the far face held at $P_{\mathrm{res}}$ in notebook 02.
**Closed model:** the last tank has no such connection. This is the sealed edge of extension C2 in notebook 02.

In the code, `SA` holds the storages $S_m A_i$ and `sig` the conductances $\sigma_i$. A model with $N$ tanks has $N$ storages and $N-1$ conductances (closed) or $N$ conductances (open).

### B1.1 Write the tank balance

Fill the two lines: Darcy between tanks, and the balance of each tank.

In [ ]:
def tank_rates(P, W, SA, sig, P0, is_open):
    '''
    dP/dt [Pa/s] of every tank.
    P    pressures of the tanks [Pa], tank 1 first
    W    production from tank 1 [kg/s]
    SA   storage of each tank, S_m*A_i [kg/Pa]
    sig  conductances [kg/(s Pa)]: sig[0] joins tanks 1 and 2, and so on;
         in an open model the last one joins the last tank to the boundary at P0
    '''
    N = len(SA)
    P_next = np.append(P[1:], P0)       # pressure on the far side of each conductance
    w = np.zeros(N + 1)                 # w[0] = W, w[i] = flow from tank i+1 into tank i
    w[0] = W
    n = N if is_open else N - 1         # number of conductances
    w[1:n + 1] = ___  # Darcy between tanks
    return ___  # tank balance: (flow in - flow out) / storage

In [ ]:
# tests
P0 = 50 * bar
# one closed tank producing 10 kg/s with storage 1e4 kg/Pa: pressure falls at 1e-3 Pa/s
assert np.isclose(tank_rates(np.array([P0]), 10.0, np.array([1e4]), np.array([]), P0, False)[0], -1e-3)
# one open tank 1 bar below P0, no production: water flows in at 1e-4 * 1e5 = 10 kg/s
assert np.isclose(tank_rates(np.array([P0 - bar]), 0.0, np.array([1e4]), np.array([1e-4]), P0, True)[0], 1e-3)
# two closed tanks, no production: the mass that leaves tank 2 enters tank 1
r = tank_rates(np.array([P0 - bar, P0]), 0.0, np.array([1e4, 1e5]), np.array([1e-4]), P0, False)
assert np.allclose(r, [1e-3, -1e-4]) and np.isclose(np.sum(np.array([1e4, 1e5]) * r), 0.0)
print("tank_rates passes")

### A1.2 Stepping in time

`simulate()` runs the model through a production history. The data files give the mean production over each interval between two times (`W[k]` is the mean between `t[k-1]` and `t[k]`).

`simulate()` first calls your `tank_rates()` once for each tank to find how the rate of every tank depends on each pressure. Then it steps forward in time. Each step uses the pressures at the end of the step (an implicit step), which stays stable for any step length. A step is at most one tenth of the shortest interval between data points; for the Hofsstaðir models, 2000 steps in every interval change the result by less than 0.02 m. You do not need to change this cell.

In [ ]:
def simulate(t, W, params, N, is_open, P0, nsub=None):
    '''
    Pressure in tank 1 [Pa] at the times t [s].
    params: [SA_1..SA_N, sig_1..sig_n] (n = N - 1 closed, N open)
    W[k]: mean production between t[k-1] and t[k] [kg/s]. All tanks start at P0.
    Steps: at most 1/10 of the shortest data interval, at least 10 per interval;
    nsub fixes the number of steps per interval instead (used for log-spaced times).
    '''
    t = np.asarray(t, float); W = np.asarray(W, float)
    SA, sig = np.asarray(params[:N], float), np.asarray(params[N:], float)
    # rates are linear in P and W: rates(P, W) = M @ P + c0 + d * W
    zero = np.zeros(N)
    c0 = tank_rates(zero, 0.0, SA, sig, P0, is_open)
    d = tank_rates(zero, 1.0, SA, sig, P0, is_open) - c0
    M = np.column_stack([tank_rates(np.eye(N)[j], 0.0, SA, sig, P0, is_open) - c0 for j in range(N)])
    P = np.full(N, float(P0))
    out = np.empty(len(t)); out[0] = P0
    inverses = {}
    h_max = np.min(np.diff(t)) / 10
    for k in range(1, len(t)):
        dt = t[k] - t[k - 1]
        n_steps = nsub if nsub else max(10, int(np.ceil(dt / h_max - 1e-9)))
        h = dt / n_steps
        key = round(h, 6)
        if key not in inverses:
            inverses[key] = np.linalg.inv(np.eye(N) - h * M)
        f = h * (c0 + d * W[k])
        for _ in range(n_steps):
            P = inverses[key] @ (P + f)     # implicit step: P_new = P_old + h * rates(P_new)
        out[k] = P[0]
    return out

### B1.3 One closed tank and one open tank

For a constant production $W$ from $t = 0$:

**Closed tank.** The pressure falls along a straight line:

$$P_1 = P_0 - \frac{W}{S_m A}\, t$$

- $P_0$: initial pressure [Pa]
- $W$: production [kg/s]
- $S_m A$: storage of the tank [kg/Pa]
- $t$: time since production started [s]

**Open tank.** The pressure falls and then levels off, because inflow from the boundary grows until it equals the production. The final drawdown and the time scale are

$$\Delta P_\infty = \frac{W}{\sigma} \qquad \tau = \frac{S_m A}{\sigma}$$

- $\Delta P_\infty$: drawdown when inflow equals production [Pa]
- $\sigma$: conductance between the tank and the boundary [kg/(s Pa)]
- $\tau$: time scale [s]. At $t = \tau$ the drawdown is $1 - e^{-1} = 63$ % of $\Delta P_\infty$.

*Prediction:* with $W$ = 20 kg/s, $S_m A$ = 1e4 kg/Pa and $\sigma$ = 2e-4 kg/(s Pa): how many bar does the closed tank lose per year? What is $\Delta P_\infty$ in bar, and $\tau$ in years?

In [ ]:
W0, SA0, sig0 = 20.0, 1e4, 2e-4
t = np.linspace(0, 5 * yr, 61)
Wc = np.full(len(t), W0)
P_closed = simulate(t, Wc, [SA0], 1, False, 0.0)
P_open = simulate(t, Wc, [SA0, sig0], 1, True, 0.0)

slope_expected = ___  # closed tank: rate of pressure change [Pa/s]
dP_inf = ___  # open tank: final drawdown [Pa]
tau = ___  # open tank: time scale [s]

slope = (P_closed[-1] - P_closed[0]) / (t[-1] - t[0])
assert np.isclose(slope, slope_expected, rtol=1e-6)
assert np.isclose(-np.interp(tau, t, P_open) / dP_inf, 1 - np.exp(-1), atol=0.01)
print(f"closed: {slope * yr / bar:.2f} bar/yr   open: final drawdown {dP_inf / bar:.2f} bar, tau = {tau / yr:.2f} yr")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t / yr, P_closed / bar, label="closed tank")
ax.plot(t / yr, P_open / bar, label="open tank")
ax.axhline(-dP_inf / bar, color="gray", ls=":", label=r"$-W/\sigma$")
ax.axvline(tau / yr, color="gray", ls="--", label=r"$\tau$")
ax.set_xlabel("time (yr)"); ax.set_ylabel(r"$P_1 - P_0$ (bar)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### B1.4 The ring model of notebook 02 is a tank model with 250 tanks

Ring $i$ lies between the faces $r_{f,i}$ and $r_{f,i+1}$, with its centre at $r_{c,i} = \sqrt{r_{f,i}\, r_{f,i+1}}$. As a tank it has

$$S_m A_i = S_m \, \pi \left( r_{f,i+1}^2 - r_{f,i}^2 \right) \qquad \sigma_i = \frac{2\pi T}{\ln\left(r_{c,i+1}/r_{c,i}\right)}$$

- $r_{f,i}$: radius of face $i$ [m]
- $r_{c,i}$: radius of the centre of ring $i$ [m]
- $T = kh/\nu$: transmissivity [m s]
- $\sigma_i$: Darcy for radial flow integrated from one ring centre to the next, as in Lecture 6.

Below: a confined reservoir of radius $R$ = 5 km with $kh$ = 10 darcy-m, either sealed at $R$ or held at $P_0$ at $R$. The well produces 20 kg/s.

*Prediction:* late in the record, which tank model should the sealed reservoir look like, and which the reservoir held at $P_0$?

In [ ]:
darcy = 9.869e-13                     # m2
kh = 10 * darcy                       # m3
T = kh / nu90                         # transmissivity [m s]
phi, c_t, h = 0.1, 1e-9, 500.0        # porosity [-], total compressibility [1/Pa] (stated), thickness [m]
S_m = rho90 * phi * c_t * h           # mass storativity [kg/(m2 Pa)]
R, r_w, n = 5000.0, 0.1, 250

rf = np.geomspace(r_w, R, n + 1)      # ring faces
rc = np.sqrt(rf[:-1] * rf[1:])        # ring centres
SA_ring = ___  # storage of each ring [kg/Pa]
sig_ring = ___  # Darcy between ring centres [kg/(s Pa)]
sig_edge = 2 * np.pi * T / np.log(R / rc[-1])     # last ring centre to the edge held at P0

assert np.isclose(SA_ring.sum(), S_m * np.pi * (R**2 - r_w**2))
assert np.isclose(np.sum(1 / sig_ring) + 1 / sig_edge, np.log(R / rc[0]) / (2 * np.pi * T))

t = np.r_[0, np.geomspace(minute, 30 * yr, 200)]
Wc = np.full(len(t), 20.0)
P_sealed = simulate(t, Wc, np.r_[SA_ring, sig_ring], n, False, 0.0, nsub=5)
P_held = simulate(t, Wc, np.r_[SA_ring, sig_ring, sig_edge], n, True, 0.0, nsub=5)

# what the tank models say about the late time
slope_closed = -20.0 / (S_m * np.pi * R**2)               # one closed tank with the whole storage
dP_open = 20.0 * np.log(R / rc[0]) / (2 * np.pi * T)      # one open tank with sigma = 2 pi T / ln(R/r_c1)
late = t > 20 * yr
assert np.isclose(np.polyfit(t[late], P_sealed[late], 1)[0], slope_closed, rtol=0.01)
assert np.isclose(-P_held[-1], dP_open, rtol=0.01)
theis = -20.0 / (4 * np.pi * T) * exp1(S_m * rc[0]**2 / (4 * T * t[1:]))

print(f"T = {T:.2e} m s, S_m = {S_m:.2e} kg/(m2 Pa), R^2 S_m / T = {R**2 * S_m / T / yr:.1f} yr")
fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogx(t[1:] / day, -P_sealed[1:] / bar, label="rings, sealed at R")
ax.semilogx(t[1:] / day, -P_held[1:] / bar, label="rings, held at $P_0$ at R")
ax.semilogx(t[1:] / day, -theis / bar, "k--", label="Theis (Lecture 6)")
ax.axhline(dP_open / bar, color="C1", ls=":", label=r"open tank: $W/\sigma$")
ax.set_ylim(0, 2.5 * dP_open / bar)
ax.set_xlabel("time (days)"); ax.set_ylabel("drawdown in the first ring (bar)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

**What Part 1 shows.** Early in the record the rings follow Theis: the funnel has not reached the edge, and the slope against log time gives $T$ (Lecture 6). Late in the record the edge controls the response. Sealed: the pressure falls at $W/(S_m \pi R^2)$, as one closed tank holding the whole storage. Held at $P_0$: the pressure levels off at $W \ln(R/r_{c,1})/(2\pi T)$, as one open tank with $\sigma = 2\pi T/\ln(R/r_{c,1})$. A lumped model describes this late part with a few tanks, and it can follow a production rate that changes every month.

### A1.5 What storage and conductance do

One open tank with $W$ = 20 kg/s, starting from the case in B1.3. The left panel changes the storage and keeps $\sigma$. The right panel changes $\sigma$ and keeps the storage. The dashed lines start with the slope $W/(S_m A)$ and reach $-W/\sigma$ at $t = \tau$.

*Prediction:* double the storage. What happens to the start slope, to the final drawdown and to $\tau$? Then double $\sigma$ instead.

In [ ]:
t5 = np.linspace(0, 5 * yr, 241)                     # [s]
W5 = np.full(len(t5), W0)                            # 20 kg/s
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for f in [0.5, 1, 2]:
    SA = f * SA0                                     # storage changes, conductance fixed [kg/Pa]
    ax[0].plot(t5 / yr, simulate(t5, W5, [SA, sig0], 1, True, 0.0) / bar,
               label=f"S_m A = {SA:.0e} kg/Pa, tau = {SA / sig0 / yr:.1f} yr")
    ax[0].plot([0, SA / sig0 / yr], [0, -W0 / sig0 / bar], "k--", lw=0.8)   # start slope W/(S_m A)
    sig = f * sig0                                   # conductance changes, storage fixed [kg/(s Pa)]
    ax[1].plot(t5 / yr, simulate(t5, W5, [SA0, sig], 1, True, 0.0) / bar,
               label=f"sigma = {sig * bar:.0f} (kg/s)/bar, W/sigma = {W0 / sig / bar:.1f} bar")
ax[1].plot(t5 / yr, simulate(t5, W5, [SA0], 1, False, 0.0) / bar, "k:", label="sigma = 0 (closed)")
ax[0].set_title("storage changes, sigma = 20 (kg/s)/bar")
ax[1].set_title("sigma changes, S_m A = 1e4 kg/Pa")
for a in ax:
    a.set_xlabel("time (yr)"); a.legend(fontsize=10, loc="lower left")
ax[0].set_ylabel(r"$P_1 - P_0$ (bar)"); ax[0].set_ylim(-3.4, 0.05)
plt.show()

Optional: change both parameters with sliders (works in Colab and Jupyter).

In [ ]:
try:
    from ipywidgets import interact, FloatLogSlider
except ImportError:
    interact = None
    print("ipywidgets is not installed; skip this cell")

if interact is not None:
    def show_tank(SA=1e4, sigma=20.0):
        # one open tank at 20 kg/s; SA in kg/Pa, sigma in (kg/s)/bar
        P = simulate(t5, W5, [SA, sigma / bar], 1, True, 0.0)
        plt.figure(figsize=(9, 4))
        plt.plot(t5 / yr, P / bar)
        plt.axhline(-W0 / sigma, color="gray", ls=":")
        plt.ylim(-7, 0.1); plt.xlabel("time (yr)"); plt.ylabel(r"$P_1 - P_0$ (bar)")
        plt.title(f"tau = {SA / (sigma / bar) / yr:.2f} yr,   W/sigma = {W0 / sigma:.2f} bar")
        plt.show()

    interact(show_tank,
             SA=FloatLogSlider(value=1e4, base=10, min=3, max=5, step=0.05, description="S_m A"),
             sigma=FloatLogSlider(value=20, base=10, min=0.5, max=2, step=0.05, description="sigma"))

## Part 2. Inverse modelling: fit a tank model to the 1997 production test at Hofsstaðir

**Forward problem:** parameters in, pressure out. That is `simulate()`.
**Inverse problem:** measured pressure in, parameters out. We search for the parameters whose `simulate()` output is closest to the data.

Hofsstaðir is a low-temperature field near Stykkishólmur, West Iceland, with water at about 87 C. Well HO-01 has supplied the Stykkishólmur district heating since late 1999. The data are water levels in HO-01, measured as depth below the wellhead, and the production of HO-01 (Guo, Axelsson, Ye and Xin, 2010, Proceedings World Geothermal Congress). A monitoring well 800 m away follows the same trend, so the level follows the pressure of the tank around the well. A fall of the water level by $\Delta h$ is a pressure drop $\rho g \Delta h$. Production in L/s becomes kg/s with the same density. We work with the pressure change from the start, so $P_0 = 0$.

In [ ]:
T_HOF_C = 87.0                                  # reservoir temperature at Hofsstadir (Guo et al. 2010) [C]
YEAR0 = 1997 + 77 / 365.25                      # 19.03.1997, the start of the record, as a decimal year
rho, _ = water(1.0, T_HOF_C)
hof = pd.read_csv("data/Hofsstadir_lumped_1997-2007.csv", comment="#")
t_all = hof["t_d"].values * day                 # [s]
W_all = hof["Q_Ls"].values * rho / 1000         # [kg/s]
level = hof["level_m"].values                   # [m below wellhead]
P_all = -rho * g * (level - level[0])           # pressure change [Pa]

test = hof["t_d"].values <= 141                 # the 1997 production test
t1, W1, P1 = t_all[test], W_all[test], P_all[test]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t1 / day, level[test], "k.", ms=4); ax[0].invert_yaxis(); ax[0].set_ylabel("water level (m)")
ax[1].step(t1 / day, W1, where="pre"); ax[1].set_ylabel("W (kg/s)"); ax[1].set_xlabel("days since 19.03.1997")
plt.show()

### B2.1 The misfit

The misfit is the root mean square of the differences between model and data:

$$\mathrm{RMS} = \sqrt{\frac{1}{n}\sum_{j=1}^{n}\left(P_{\mathrm{model},j} - P_{\mathrm{obs},j}\right)^2}$$

- $n$: number of measured points
- $P_{\mathrm{model},j}$: model pressure at the time of point $j$ [Pa]
- $P_{\mathrm{obs},j}$: measured pressure at point $j$ [Pa]

For a water level record, the pressures are converted to water levels with $\Delta h = \Delta P/(\rho g)$, so dividing the RMS by $\rho g$ gives it in metres of water level.

In [ ]:
def misfit(params, N, is_open, t, W, Pobs, P0=0.0, mask=None):
    '''RMS misfit in metres of water level. mask selects the points that count.'''
    if mask is None:
        mask = np.isfinite(Pobs)
    Pm = simulate(t, W, params, N, is_open, P0)
    return ___  # RMS misfit in metres

# tests: a model compared with itself has zero misfit; a 1 m offset gives 1 m
Pm = simulate(t1, W1, [200, 6e-5], 1, True, 0.0, nsub=10)
assert np.isclose(misfit([200, 6e-5], 1, True, t1, W1, Pm), 0.0)
assert np.isclose(misfit([200, 6e-5], 1, True, t1, W1, Pm + rho * g), 1.0)
print("misfit passes")

### B2.2 Three guesses with one open tank

*Prediction:* which of the three parameter pairs below fits the test best? Look at the data plot above and think about $\Delta P_\infty = W/\sigma$ and $\tau = S_m A/\sigma$.

In [ ]:
guesses = [(1000, 3e-5), (200, 6e-5), (50, 1e-4)]       # (S_m A [kg/Pa], sigma [kg/(s Pa)])
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t1 / day, -P1 / (rho * g), "k.", ms=4, label="data")
for SA, s in guesses:
    J = misfit([SA, s], 1, True, t1, W1, P1)
    Pm = simulate(t1, W1, [SA, s], 1, True, 0.0)
    ax.plot(t1 / day, -Pm / (rho * g), label=f"SA = {SA}, sigma = {s:.0e}: RMS {J:.1f} m")
ax.invert_yaxis(); ax.set_xlabel("days"); ax.set_ylabel("drawdown (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### B2.3 Grid search

Compute the misfit at every point of a grid in $\log_{10} S_m A$ and $\log_{10}\sigma$. The map of the misfit over the grid is the misfit surface. Its lowest point is the best fit of the one-tank open model.

In [ ]:
SA_grid = np.logspace(1, 4, 41)
sig_grid = np.logspace(-5.5, -3.5, 31)
J = np.empty((len(sig_grid), len(SA_grid)))
for i, s in enumerate(sig_grid):
    for j, SA in enumerate(SA_grid):
        J[i, j] = ___  # misfit at this grid point

i, j = np.unravel_index(np.argmin(J), J.shape)
best_grid = (SA_grid[j], sig_grid[i])
assert 100 < best_grid[0] < 400 and 4e-5 < best_grid[1] < 9e-5
print(f"{J.size} model runs. Lowest misfit {J[i, j]:.2f} m at S_m A = {best_grid[0]:.0f} kg/Pa, "
      f"sigma = {best_grid[1]:.1e} kg/(s Pa) = {best_grid[1] * bar:.1f} (kg/s)/bar")

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(np.log10(SA_grid), np.log10(sig_grid), np.log10(J), 20, cmap="viridis")
plt.colorbar(cs, label=r"$\log_{10}$ RMS (m)")
ax.plot(np.log10(best_grid[0]), np.log10(best_grid[1]), "w+", ms=14, mew=2)
ax.set_xlabel(r"$\log_{10}\, S_m A$ (kg/Pa)"); ax.set_ylabel(r"$\log_{10}\, \sigma$ (kg/(s Pa))")
plt.show()

### A2.4 Let the computer search

`least_squares` starts from a guess. It changes each parameter a little to find which way the misfit falls, steps that way, and repeats until no step lowers the misfit. It works on $\log_{10}$ of the parameters, because storage and conductance can lie anywhere over several orders of magnitude. The white line shows every point it tried.

In [ ]:
def fit(params0, N, is_open, t, W, Pobs, P0=0.0, mask=None, path=None):
    '''Least squares on log10 of the parameters. Returns the best parameters.'''
    if mask is None:
        mask = np.isfinite(Pobs)
    nsa = N
    lo = np.r_[np.full(nsa, -1.0), np.full(len(params0) - nsa, -14.0)]   # bounds on log10
    hi = np.r_[np.full(nsa, 13.0), np.full(len(params0) - nsa, 1.0)]
    def resid(x):
        if path is not None:
            path.append(x.copy())
        return simulate(t, W, 10**x, N, is_open, P0)[mask] - Pobs[mask]
    x0 = np.clip(np.log10(params0), lo + 1e-6, hi - 1e-6)
    r = least_squares(resid, x0, bounds=(lo, hi), diff_step=1e-4)
    return 10**r.x

path = []
best_ls = fit([1000, 1e-5], 1, True, t1, W1, P1, path=path)
path = np.array(path)
print(f"{len(path)} model runs. S_m A = {best_ls[0]:.0f} kg/Pa, sigma = {best_ls[1]:.2e} kg/(s Pa), "
      f"RMS = {misfit(best_ls, 1, True, t1, W1, P1):.2f} m")

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(np.log10(SA_grid), np.log10(sig_grid), np.log10(J), 20, cmap="viridis")
plt.colorbar(cs, label=r"$\log_{10}$ RMS (m)")
ax.plot(path[:, 0], path[:, 1], "w.-", ms=4, lw=0.8)
ax.set_xlim(1, 4); ax.set_ylim(-5.5, -3.5)
ax.set_xlabel(r"$\log_{10}\, S_m A$ (kg/Pa)"); ax.set_ylabel(r"$\log_{10}\, \sigma$ (kg/(s Pa))")
plt.show()

### B2.5 Test the method with a known answer

Make data from known parameters, add random noise, and fit. If the search works, it returns parameters close to the known ones. The noise here is 2.4 m, the size of the misfit of the one-tank fit to the real data. Repeat with 20 different noise draws (the Monte Carlo idea of Lecture 8) to see how far noise of that size moves the answer.

Compare the fitted curve with the data in A2.4 or A2.6: are the real differences random, like this noise, or do they have a pattern?

In [ ]:
rng = np.random.default_rng(4)
true = np.array([200.0, 6e-5])
noise_m = 2.4
P_true = simulate(t1, W1, true, 1, True, 0.0)
found = []
for k in range(20):
    P_syn = ___  # add noise of noise_m metres, in Pa
    found.append(fit([1000, 1e-5], 1, True, t1, W1, P_syn))
found = np.array(found)
assert np.all(np.abs(found / true - 1) < 0.3)
print("spread of the fitted values (min, max):")
print(f"  S_m A: {found[:, 0].min():.0f} to {found[:, 0].max():.0f} kg/Pa   (true {true[0]:.0f})")
print(f"  sigma: {found[:, 1].min():.2e} to {found[:, 1].max():.2e}   (true {true[1]:.0e})")

### A2.6 More tanks on the same test

The one-tank open model misses the first days and the recovery. Fit models with more tanks. Each step on the ladder starts from the previous fit: a closed model becomes open by adding a conductance to the boundary; an open model becomes closed with one more tank by turning the boundary into a tank ten times larger than the last one.

In [ ]:
def model_name(N, is_open):
    return f"{N} tank {'open' if is_open else 'closed'}"

def next_model(params, N, is_open, t_span):
    '''Starting guess for the next model on the ladder 1c, 1o, 2c, 2o, 3c, 3o.'''
    SA, sig = params[:N], params[N:]
    if not is_open:
        return np.r_[SA, sig, SA[-1] / (10 * t_span)], N, True
    return np.r_[SA, 10 * SA[-1], sig], N + 1, False

def closed1_start(t, W, Pobs, P0, mask):
    '''One closed tank: P0 - P = (produced mass) / (S_m A), solved directly.'''
    V = np.r_[0, np.cumsum(W[1:] * np.diff(t))]
    dP = P0 - Pobs
    return np.array([np.sum(V[mask]**2) / np.sum(V[mask] * dP[mask])])

def ladder(t, W, Pobs, P0=0.0, mask=None, models=6):
    if mask is None:
        mask = np.isfinite(Pobs)
    out = {}
    x, N, op = closed1_start(t, W, Pobs, P0, mask), 1, False
    for k in range(models):
        p = fit(x, N, op, t, W, Pobs, P0, mask)
        out[(N, op)] = p
        if k < models - 1:
            x, N, op = next_model(p, N, op, t[-1] - t[0])
    return out

test_fits = ladder(t1, W1, P1, models=4)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t1 / day, -P1 / (rho * g), "k.", ms=4, label="data")
for (N, op), p in test_fits.items():
    J_ = misfit(p, N, op, t1, W1, P1)
    ax.plot(t1 / day, -simulate(t1, W1, p, N, op, 0.0) / (rho * g), label=f"{model_name(N, op)}: RMS {J_:.2f} m")
ax.invert_yaxis(); ax.set_xlabel("days"); ax.set_ylabel("drawdown (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

## Part 3. The Hofsstaðir record 1997 to 2007: which model, which forecast?

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(YEAR0 + t_all / yr, level, "k.", ms=3); ax[0].invert_yaxis(); ax[0].set_ylabel("water level (m)")
ax[1].step(YEAR0 + t_all / yr, W_all, where="pre"); ax[1].set_ylabel("W (kg/s)")
plt.show()

### B3.1 The model ladder and the standard deviation

Each tank adds parameters, and the misfit can only fall when parameters are added. To compare models with different numbers of parameters, divide by the degrees of freedom instead of the number of points:

$$s = \sqrt{\frac{1}{n - p}\sum_{j=1}^{n}\left(P_{\mathrm{model},j} - P_{\mathrm{obs},j}\right)^2}$$

- $s$: estimated standard deviation of the data about the model [Pa, or m of water after dividing by $\rho g$]
- $n$: number of measured points
- $p$: number of model parameters
- $n - p$: degrees of freedom

In [ ]:
def std_dev(params, N, is_open, t, W, Pobs, P0=0.0, mask=None):
    '''Standard deviation about the model, metres of water level.'''
    if mask is None:
        mask = np.isfinite(Pobs)
    res = simulate(t, W, params, N, is_open, P0)[mask] - Pobs[mask]
    n, p = mask.sum(), len(params)
    return ___  # standard deviation in metres

hof_fits = ladder(t_all, W_all, P_all)
rows = []
for (N, op), p in hof_fits.items():
    rows.append(dict(model=model_name(N, op), p=len(p),
                     RMS_m=misfit(p, N, op, t_all, W_all, P_all),
                     s_m=std_dev(p, N, op, t_all, W_all, P_all),
                     sigma1_kgs_per_bar=p[N] * bar if len(p) > N else np.nan,
                     storage_t_per_m=p[:N].sum() * rho * g / 1000))
table = pd.DataFrame(rows)
assert table["RMS_m"].iloc[3] < 2.5 and table["s_m"].iloc[3] > table["RMS_m"].iloc[3]
table.round(2)

*Prediction:* the two-tank open and the three-tank closed models fit almost equally well. Continue production at 20 L/s until the end of 2038, the case Guo et al. (2010) forecast. Which model forecasts the lower water level in 2038, and by roughly how much?

### B3.2 Forecast

In [ ]:
W_future = ___  # 20 L/s as kg/s
n_f = int((2039.0 - (YEAR0 + t_all[-1] / yr)) * yr / (10 * day))   # 10-day steps to the end of 2038
t_f = np.r_[t_all, t_all[-1] + np.arange(1, n_f + 1) * 10 * day]
W_f = np.r_[W_all, np.full(n_f, W_future)]
assert np.isclose(W_future, 19.35, atol=0.05)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(YEAR0 + t_all / yr, level, "k.", ms=3, label="data")
for key in [(2, True), (3, False)]:
    Pf = simulate(t_f, W_f, hof_fits[key], *key, 0.0)
    lev = level[0] - Pf / (rho * g)
    ax.plot(YEAR0 + t_f / yr, lev, label=f"{model_name(*key)}: {lev[-1]:.0f} m in 2038")
ax.invert_yaxis(); ax.set_xlabel("year"); ax.set_ylabel("water level (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### B3.3 Fit part of the record, predict the rest

A forecast can be tested on data the model has not seen. Fit only the data before spring 2003 (day 2200) and predict 2003 to 2007.

*Prediction:* will the model with the smallest misfit before 2003 also give the best prediction after 2003?

In [ ]:
before = hof["t_d"].values < 2200
after = ~before
fits_03 = ladder(t_all, W_all, P_all, mask=before)
rows = []
for (N, op), p in fits_03.items():
    Pm = simulate(t_all, W_all, p, N, op, 0.0)
    rms_pred = ___  # prediction misfit after 2003 [m]
    rows.append(dict(model=model_name(N, op), p=len(p),
                     RMS_fit_m=misfit(p, N, op, t_all, W_all, P_all, mask=before), RMS_predict_m=rms_pred))
val = pd.DataFrame(rows)
assert val["RMS_predict_m"].min() < 5
val.round(2)

### B3.4 What the storage and the conductance say about the reservoir

The storage of a tank is $S_m A$. Its area depends on how the reservoir stores water:

$$S_m = \rho\,\phi\, c_t\, h \quad \text{(confined)} \qquad S_m = \frac{\phi}{g} \quad \text{(free surface)}$$

- $\phi$: porosity [-]
- $c_t$: total compressibility of rock and water [1/Pa]
- $h$: reservoir thickness [m]
- $g$: gravitational acceleration [m/s²]. Lowering a free surface by $\Delta h$ drains $\rho \phi A \Delta h$ of water while the pressure falls by $\rho g \Delta h$, so $S_m = \phi/g$.

The conductance between tank 1 and tank 2 gives $kh$ if the flow between them is radial, with tank 1 a disc of radius $R/2$ and tank 2 a ring around it centred on $3R/2$:

$$\sigma_1 = \frac{2\pi\, kh}{\nu \ln(r_2/r_1)} \qquad r_1 = R/2, \quad r_2 = 3R/2$$

- $k$: permeability [m²]
- $h$: reservoir thickness [m]
- $\nu$: kinematic viscosity of the water [m²/s]
- $r_1, r_2$: radii of the two tanks' centres [m]; $\ln(r_2/r_1) = \ln 3$ whatever $R$ is

In [ ]:
phi, c_t, h_res = 0.1, 1e-9, 1000.0
SA_total = hof_fits[(3, False)][:3].sum()
S_m_conf = ___  # confined storage [kg/(m2 Pa)]
S_m_free = ___  # free-surface storage [kg/(m2 Pa)]
assert np.isclose(S_m_free / S_m_conf, 1 / (rho * g * c_t * h_res))
print(f"total storage of the 3-tank closed model: {SA_total:.2e} kg/Pa")
print(f"area if confined:     {SA_total / S_m_conf / 1e6:8.1f} km2")
print(f"area if free surface: {SA_total / S_m_free / 1e6:8.2f} km2")
_, nu_hof = water(1.0, T_HOF_C)
sig1 = hof_fits[(3, False)][3]                           # conductance between tanks 1 and 2 [kg/(s Pa)]
kh_hof = ___  # kh from sigma_1 [m3]
assert 3 < kh_hof / darcy < 6
print(f"sigma_1 = {sig1 * bar:.1f} (kg/s)/bar  gives  kh = {kh_hof / darcy:.1f} darcy-m")

### Reflection

1. Which part of the Hofsstaðir record decides between the open and the closed model? Could more years of data settle it?
2. In B3.3, which model had the smallest misfit before 2003, and which predicted best after 2003? What does that tell you about adding tanks?
3. The storage gives an area only after you choose a storage mechanism. Which mechanism do you think applies at Hofsstaðir, and why?
4. Guo et al. (2010, Fig. 11) forecast the same case with LUMPFIT, a 2-tank open and a 3-tank closed model. Compare your two forecasts for 2038 with theirs.
5. Name one thing that could make both forecasts wrong.

## Extensions (C)

**C1 Gata, initial level unknown.** The Gata file is from the Laugaland field in Holt, South Iceland, with water at about 100 C. It holds the water level in well GN-1 at Gata and the production at Laugaland, 1982 to 2000 (Axelsson, 1989). The file has no water level before week 65 (mid-1983), so the initial level $P_0$ must be fitted as well. Write a version of `fit()` that adds $P_0$ to the parameters (not in log10) and run the ladder. How well is $P_0$ determined?

**C2 Largest production.** Find the largest constant production that keeps the water level above 200 m until 2038, for the two-tank open and the three-tank closed models.

**C3 Fit the ring model.** Use `P_sealed` from B1.4 (sampled monthly for 10 years, with 0.05 bar noise) as data and fit tank models. Compare the total storage with $S_m \pi R^2$ and the sum of $1/\sigma_i$ with $\ln(R/r_{c,1})/(2\pi T)$. Which quantities does the fit recover, and which not?

In [ ]:
# C1 to C3: your code here

## Assignment 4: Ahuachapán

The file `Ahuachapan_lumped_1968-2001.csv` holds the reservoir pressure (bar gauge) and the net mass production (kg/s) of the Ahuachapán field, El Salvador, month by month from 1968. The pressure before production was 36.0 bar.

Hand in a short written report (PDF) with equations, numbers and figures, and your notebook. Define every symbol after each equation, with units.

1. Fit the six models of the ladder to the whole record. Give a table of $p$, RMS and $s$ in bar. How much does $s$ fall with each added tank, and where does adding a tank stop paying off?
2. Fit the 1-tank open, 2-tank closed and 2-tank open models to the data up to the end of 1985 only. Forecast the pressure to 2001 with the production that was measured, and compare with the measured pressure. Which model would you have trusted in 1985, and why?
3. Take the total storage of the 2-tank closed model fitted to the whole record. Compute the area it implies for confined liquid storage ($\phi$ = 0.1, $c_t$ = 1e-9 1/Pa, $h$ = 1000 m, water density at 230 C from `water()`) and for two-phase storage (take $c_t$ 1000 times larger). Compare both with the size of a geothermal field, a few to a few tens of km². Which explanation fits: two-phase storage, recharge from outside, or both?
4. Forecast the pressure in 2021 with the last production rate held constant, with the 2-tank closed and the 2-tank open models. In five to ten sentences, say which forecast you would give the field operator and one way the model could mislead them.